# Algorithm 3.3 — The universal Kepler's equation

**Goal:** one equation to replace both 3.1 and 3.2. Given the initial radius $r_0$, initial radial speed $v_{r0}$, the reciprocal semimajor axis $\alpha=1/a$, and an elapsed time $\Delta t$, solve the **universal Kepler's equation** for the **universal anomaly** $\chi$ — for an ellipse, parabola, or hyperbola, with the *same* code.

**Code (answer key):** [`kepler_U`](../curtis/algorithms/alg_3_3_kepler_U.py) · **Book:** §3.7, Algorithm 3.3 · **Worked example:** 3.6

This is where the Stumpff functions $C(z), S(z)$ earn their keep — they carry the conic type so the equation never has to branch.

## Read first

| Symbol | Name | Units |
|---|---|---|
| $\chi$ | universal anomaly (the unknown) | km$^{1/2}$ |
| $z = \alpha\,\chi^2$ | Stumpff argument; sign = conic type | — |
| $\alpha = 1/a$ | reciprocal of the semimajor axis | 1/km |
| $r_0$ | radius at $\chi=0$ | km |
| $v_{r0}$ | radial speed at $\chi=0$ | km/s |
| $\Delta t$ | elapsed time | s |

**Universal Kepler's equation (Eq 3.62):**
$$\sqrt{\mu}\,\Delta t=\frac{r_0 v_{r0}}{\sqrt{\mu}}\,\chi^2 C(z)+(1-\alpha r_0)\,\chi^3 S(z)+r_0\,\chi,\qquad z=\alpha\chi^2.$$

> Note the footgun: the fourth argument $\alpha$ is the **reciprocal** of the semimajor axis (the book calls it `a`). Pass `1/a`, not `a`.

## The picture (one anomaly for every orbit)

3.1 used the eccentric anomaly $E$; 3.2 used $F$; a parabola would need yet another variable. The **universal anomaly $\chi$** replaces all three. It is defined by $\dfrac{d\chi}{dt}=\dfrac{\sqrt{\mu}}{r}$, and it reduces to the old anomalies case by case:

| orbit | $z=\alpha\chi^2$ | $\chi$ reduces to |
|---|---|---|
| ellipse | $>0$ | $\sqrt{a}\,(E-E_0)$ |
| parabola | $0$ | $\propto \tan(\theta/2)$ |
| hyperbola | $<0$ | $\sqrt{-a}\,(F-F_0)$ |

Because $C(z)$ and $S(z)$ are smooth through $z=0$, the single equation above is valid across all three — **no `if e<1` branching**. You solve it for $\chi$ with Newton's method, exactly as before.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys; sys.path.insert(0, "..")
# Reuse the Stumpff functions from the previous notebook (importing the
# reference here so this notebook stays focused on the universal Kepler step).
from curtis.algorithms.stumpff import stumpC, stumpS

In [ ]:
# The universal Kepler residual F(chi) for Example 3.6 (a hyperbola, a < 0).
mu, r0, vr0, dt, a_semi = 398600.0, 10000.0, 3.0752, 3600.0, -19655.0
alpha = 1/a_semi

chi = np.linspace(0, 250, 500)
z = alpha*chi**2
C = np.array([stumpC(zz) for zz in z])
S = np.array([stumpS(zz) for zz in z])
F = (r0*vr0/np.sqrt(mu))*chi**2*C + (1 - alpha*r0)*chi**3*S + r0*chi - np.sqrt(mu)*dt

chi0 = np.sqrt(mu)*abs(alpha)*dt       # Newton's starting guess
root = 128.511                          # the answer we are after

fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.axhline(0, color='#c2c0b6', lw=0.9)
ax.plot(chi, F/1e6, color='#378ADD', lw=2)
ax.axvline(root, color='#1D9E75', ls='--', lw=1)
ax.plot(root, 0, 'o', color='#1D9E75')
ax.axvline(chi0, color='#D85A30', ls=':', lw=1.2)
ax.annotate(f'root  χ = {root}', (root+4, 1.2), color='#1D9E75')
ax.annotate('start χ₀', (chi0-4, -2.0), color='#D85A30', ha='right')
ax.set_xlabel(r'universal anomaly  $\chi$  (km$^{1/2}$)')
ax.set_ylabel(r'residual  $F(\chi)\;(\times10^6)$')
ax.set_title('One equation, one root — Newton finds χ (Example 3.6, a hyperbola)')
plt.show()

## See it — a smooth residual with a single root

The blue curve is the universal Kepler equation moved all to one side,
$$F(\chi)=\frac{r_0 v_{r0}}{\sqrt{\mu}}\chi^2 C(z)+(1-\alpha r_0)\chi^3 S(z)+r_0\chi-\sqrt{\mu}\,\Delta t,$$
and the answer $\chi$ is where it crosses zero. It is smooth and monotonic, so Newton's method marches from the dotted starting guess $\chi_0$ straight to the root with no drama.

Here $a<0$, so $z=\alpha\chi^2<0$ throughout — you are on the **hyperbolic** (cosh/sinh) side of the Stumpff functions. Flip the orbit to an ellipse ($a>0$) and the *exact same* `F(χ)` would have $z>0$ and still cross zero once. That single-equation, single-root behaviour for every conic is the whole point of Algorithm 3.3.

## Where these equations come from

### The universal anomaly $\chi$
Define $\chi$ by $\dfrac{d\chi}{dt}=\dfrac{\sqrt{\mu}}{r}$ (Eq 3.56). It is a "regularised" time variable: where $r$ is small (fast motion near perigee) $\chi$ advances quickly, smoothing the equations. Integrating this relation, with $r$ written in universal variables, produces the **universal Kepler equation** (Eq 3.62)
$$\sqrt{\mu}\,\Delta t=\frac{r_0 v_{r0}}{\sqrt{\mu}}\chi^2 C(z)+(1-\alpha r_0)\chi^3 S(z)+r_0\chi.$$
The Stumpff functions $C(z),S(z)$ appear precisely because the integral $\int dt = \int r\,d\chi/\sqrt{\mu}$ generates their power series — that is *why* they were worth defining separately.

### Newton's method (and the derivative, Eq 3.65)
Move everything to one side, $f(\chi)=\text{RHS}-\sqrt{\mu}\,\Delta t$, and find the root. Differentiating (using the Stumpff identities $\tfrac{dC}{dz},\tfrac{dS}{dz}$, which collapse neatly) gives
$$f'(\chi)=\frac{r_0 v_{r0}}{\sqrt{\mu}}\,\chi\,(1-\alpha\chi^2 S)+(1-\alpha r_0)\,\chi^2 C+r_0\qquad(3.65).$$
So the update is $\chi\leftarrow\chi-f/f'$, quadratically convergent.

### Starting guess
$\chi_0=\sqrt{\mu}\,|\alpha|\,\Delta t$ (Chobotov) — a good seed for all conic types.

## Step by step (in code order)

**1. Starting guess.** $\;\chi=\sqrt{\mu}\,|\alpha|\,\Delta t$.

**2. Newton iteration** until the correction is below `tol` (with an iteration cap `nMax`). Each pass:
$$z=\alpha\chi^2,\quad C=C(z),\quad S=S(z),$$
$$f=\frac{r_0 v_{r0}}{\sqrt{\mu}}\chi^2 C+(1-\alpha r_0)\chi^3 S+r_0\chi-\sqrt{\mu}\,\Delta t,$$
$$f'=\frac{r_0 v_{r0}}{\sqrt{\mu}}\chi(1-\alpha\chi^2 S)+(1-\alpha r_0)\chi^2 C+r_0,$$
$$\chi\leftarrow\chi-f/f'.$$

**↓ Now type your implementation below.** It is a Newton loop like 3.1/3.2, but each pass first computes `z = alpha*chi**2` and calls `stumpC(z)`, `stumpS(z)`. Watch the `sqrt(mu)` placements. Answer key linked above; peek only after you try.

In [ ]:
def kepler_U(dt, ro, vro, alpha, mu, tol=1.0e-8, nMax=1000):
    """Universal anomaly chi solving the universal Kepler equation.

    alpha = 1 / (semimajor axis)  -- the RECIPROCAL (the book calls it `a`).
    """
    smu = np.sqrt(mu)

    # 1. Starting guess:  chi = sqrt(mu) * |alpha| * dt

    # 2. Newton iteration until |ratio| < tol (or n > nMax):
    #        z = alpha * chi**2
    #        C = stumpC(z);  S = stumpS(z)
    #        F    = ro*vro/smu * chi**2 * C + (1 - alpha*ro) * chi**3 * S + ro*chi - smu*dt
    #        dFdx = ro*vro/smu * chi * (1 - alpha*chi**2 * S) + (1 - alpha*ro) * chi**2 * C + ro
    #        ratio = F / dFdx
    #        chi = chi - ratio

    # return chi
    raise NotImplementedError("fill in steps 1-2, then delete this line")

## Verify — Example 3.6

**Input** ($\mu_\oplus=398600$): $\;r_0=10000$ km, $\;v_{r0}=3.0752$ km/s, $\;\Delta t=3600$ s, $\;a=-19655$ km (note $a<0$ → hyperbola).

**Expected:** $\chi = 128.511$ km$^{1/2}$.

Remember to pass the **reciprocal** `1/a`. Run once your function is typed.

In [ ]:
mu  = 398600.0
r0  = 10000.0
vr0 = 3.0752
dt  = 3600.0
a   = -19655.0                 # semimajor axis (km); negative -> hyperbola

chi = kepler_U(dt, r0, vr0, 1/a, mu)     # pass alpha = 1/a !

print(f"chi = {chi:.6g} km^0.5   (expect 128.511)")
print(f"z   = {(1/a)*chi**2:.4g}   (negative -> hyperbolic branch of C, S)")

assert abs(chi - 128.511) < 1e-3
print("\nAll checks passed ✔")

## What this confirms

- **One equation replaces 3.1 and 3.2.** The same `F(χ)` solves ellipse, parabola and hyperbola; the sign of $z=\alpha\chi^2$ routes through the right branch of $C, S$ automatically.
- $\chi$ is a smooth, well-behaved unknown with a single root — Newton converges from a cheap starting guess.
- This is the engine for **propagation**: once you can get $\chi$ from $\Delta t$, the Lagrange coefficients turn it into a new state vector.

**Next:** the **Lagrange coefficients $f, g$** (and $\dot f, \dot g$) — they package $\chi$ and the Stumpff functions into the four numbers that advance $(\mathbf{r}_0,\mathbf{v}_0)$ to $(\mathbf{r},\mathbf{v})$, completing the chain to Algorithm 3.4.